# Training code and optional rerun

This is the full original RF/MLP runner source, with only its automatic entry-point call removed. It defines the same 15-fit matrix, feature construction, weighted loss, optimizer, validation checkpoint selection, and export logic as `train.py`. The original run was executed through that script; its training histories are displayed in `01_experiment_record.ipynb`.

**Run all defaults to defining the code only.** No model is fitted unless `RUN_TRAINING` is deliberately enabled below. A new run needs the trusted private caches and frozen partitions; the public repository contains aggregate outputs only. Use a new output directory to preserve the original experiment.

Install the dependencies listed in the recorded training manifest plus Jupyter. The preparation and split logic remain in `prepare.py` and `finish.py`, displayed in the final section.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train.py").exists() and (p / "results/training_manifest.json").exists())
__file__ = str(ROOT / "train.py")

## Exact training definitions

In [2]:
"""Retrain RF and matched MLP fusion ablations using one frozen split manifest."""

import argparse, copy, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import sklearn
from sklearn.ensemble import RandomForestClassifier

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "drafting_code"))
from snooppi_utils import (
    file_sha256,
    load_embedding_cache,
    build_pair_features,
    assert_model_ready,
    require_comparable_preprocessing,
    validate_cache_coverage,
    select_threshold,
    summarize_predictions,
    macro_average_precision,
)
from snooppi_protocol import mapped_overlap_report

ENCODERS = {"prott5": "ProtT5", "esm2": "ESM-2 650M", "esmc": "ESM-C 600M"}
VARIANTS = {
    "full": ("sum", "absdiff", "product"),
    "no_sum": ("absdiff", "product"),
    "no_absdiff": ("sum", "product"),
    "no_product": ("sum", "absdiff"),
}


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--splits", type=Path, required=True)
    p.add_argument("--inputs", type=Path, required=True)
    p.add_argument("--out", type=Path, required=True)
    p.add_argument("--seed", type=int, default=44)
    p.add_argument("--resume", action="store_true")
    args = p.parse_args()
    torch.set_num_threads(2)
    torch.use_deterministic_algorithms(True)
    manifest = json.loads((args.splits / "manifest.json").read_text())
    for name, sha in manifest["files"].items():
        if file_sha256(args.splits / name) != sha:
            raise ValueError("Frozen artifact changed: " + name)
    parts = {
        n: pd.read_csv(args.splits / f"{n}.csv")
        for n in ("train", "validation", "test")
    }
    from finish import input_isolation

    input_isolation(parts)
    assert_model_ready(parts)
    require_comparable_preprocessing(parts, list(ENCODERS.values()))
    m = pd.read_csv(args.splits / "mmseqs2_clusters.csv")
    if mapped_overlap_report(parts, m, "cluster_id").overlapping_cluster_id.any():
        raise ValueError("Cluster leakage")
    # Preflight every cache before training any model.
    cache_records = {}
    for k, e in ENCODERS.items():
        c, meta = load_embedding_cache(args.inputs / f"{k}.pt", e)
        validate_cache_coverage(c, parts)
        cache_records[k] = {
            "sha256": file_sha256(args.inputs / f"{k}.pt"),
            "metadata": meta,
        }
        del c
    args.out.mkdir(parents=True, exist_ok=args.resume)
    config = {
        "seed": args.seed,
        "split_manifest_sha256": file_sha256(args.splits / "manifest.json"),
        "caches": cache_records,
        "rf": {"trees": 400, "min_samples_leaf": 2, "max_features": "sqrt"},
        "mlp": {
            "hidden": [512, 512, 256],
            "dropout": 0.1,
            "batch_size": 1024,
            "max_epochs": 50,
            "patience": 10,
            "lr": 1e-4,
            "weight_decay": 1e-4,
            "checkpoint": "validation macro AP",
        },
        "variants": VARIANTS,
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "python": sys.version,
        "script_sha256": file_sha256(Path(__file__)),
    }
    if (args.out / "manifest.json").exists():
        if json.loads((args.out / "manifest.json").read_text()) != json.loads(
            json.dumps(config)
        ):
            raise ValueError("Resume configuration changed")
    else:
        (args.out / "manifest.json").write_text(json.dumps(config, indent=2))
    y = {n: f.label.to_numpy(dtype=np.float32) for n, f in parts.items()}
    weight = float((y["train"] == 0).sum() / (y["train"] == 1).sum())
    metrics = []
    for k, e in ENCODERS.items():
        cache, _ = load_embedding_cache(args.inputs / f"{k}.pt", e)
        for variant, ops in VARIANTS.items():
            x = {n: build_pair_features(f, cache, ops) for n, f in parts.items()}
            for head in (["rf", "mlp"] if variant == "full" else ["mlp"]):
                start = time.monotonic()
                tag = f"{k}_{head}_{variant}"
                torch.manual_seed(args.seed)
                np.random.seed(args.seed)
                record_path = args.out / f"{tag}_complete.json"
                if args.resume and record_path.exists():
                    record = json.loads(record_path.read_text())
                    for name, sha in record["files"].items():
                        if file_sha256(args.out / name) != sha:
                            raise ValueError("Completed artifact changed")
                    metrics.extend(record["metrics"])
                    continue
                history = []
                best_epoch = None
                if head == "rf":
                    model = RandomForestClassifier(
                        n_estimators=400,
                        min_samples_leaf=2,
                        max_features="sqrt",
                        class_weight={0: 1.0, 1: weight},
                        random_state=args.seed,
                        n_jobs=2,
                    )
                    model.fit(x["train"], y["train"])
                    scores = {n: model.predict_proba(v)[:, 1] for n, v in x.items()}
                else:
                    layers = []
                    d = x["train"].shape[1]
                    for h in (512, 512, 256):
                        layers.extend(
                            [
                                torch.nn.Linear(d, h),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                            ]
                        )
                        d = h
                    model = torch.nn.Sequential(*layers, torch.nn.Linear(d, 1))
                    optimizer = torch.optim.AdamW(
                        model.parameters(), lr=1e-4, weight_decay=1e-4
                    )
                    loss_fn = torch.nn.BCEWithLogitsLoss(
                        pos_weight=torch.tensor(weight)
                    )
                    xt = {n: torch.from_numpy(v) for n, v in x.items()}
                    yt = torch.from_numpy(y["train"])

                    def predict(n):
                        model.eval()
                        with torch.no_grad():
                            return torch.cat(
                                [
                                    model(z).flatten().sigmoid()
                                    for z in xt[n].split(1024)
                                ]
                            ).numpy()

                    best = -float("inf")
                    stale = 0
                    for epoch in range(1, 51):
                        model.train()
                        order = torch.randperm(len(yt))
                        loss_sum = 0.0
                        for ids in order.split(1024):
                            optimizer.zero_grad()
                            loss = loss_fn(model(xt["train"][ids]).flatten(), yt[ids])
                            loss.backward()
                            optimizer.step()
                            loss_sum += loss.item() * len(ids)
                        ap = macro_average_precision(
                            y["validation"], predict("validation")
                        )
                        history.append(
                            {
                                "epoch": epoch,
                                "train_weighted_loss": loss_sum / len(yt),
                                "validation_macro_ap": ap,
                            }
                        )
                        if ap > best:
                            best = ap
                            state = copy.deepcopy(model.state_dict())
                            best_epoch = epoch
                            stale = 0
                        else:
                            stale += 1
                        if stale >= 10:
                            break
                    model.load_state_dict(state)
                    scores = {n: predict(n) for n in parts}
                    pd.DataFrame(history).to_csv(
                        args.out / f"{tag}_history.csv", index=False
                    )
                threshold = select_threshold(y["validation"], scores["validation"])
                for n in parts:
                    metrics.append(
                        {
                            "encoder": k,
                            "head": head,
                            "variant": variant,
                            "split": n,
                            "seed": args.seed,
                            "best_epoch": best_epoch,
                            "seconds": time.monotonic() - start,
                            **summarize_predictions(y[n], scores[n], threshold),
                        }
                    )
                pd.DataFrame(
                    {
                        "full_pair_id": parts["test"].full_pair_id,
                        "label": y["test"],
                        "probability": scores["test"],
                    }
                ).to_csv(args.out / f"{tag}_test_predictions.csv", index=False)
                files = [args.out / f"{tag}_test_predictions.csv"]
                if history:
                    files.append(args.out / f"{tag}_history.csv")
                record_path.write_text(
                    json.dumps(
                        {
                            "metrics": metrics[-3:],
                            "files": {p.name: file_sha256(p) for p in files},
                        },
                        indent=2,
                    )
                )
                pd.DataFrame(metrics).to_csv(args.out / "metrics.csv", index=False)
                print(
                    tag,
                    "completed",
                    round(time.monotonic() - start, 1),
                    "seconds",
                    flush=True,
                )
                del model
            del x
        del cache
    pd.DataFrame(metrics).to_csv(args.out / "metrics.csv", index=False)
    (args.out / "COMPLETE").write_text(
        "15 fresh classifier fits; one fixed split and matched seed 44.\n"
    )




## Optional fresh fit
Change paths and opt in only when a new training run is intended. The saved notebook output below records that this cell was skipped during notebook assembly.

In [3]:
RUN_TRAINING = False
INPUTS = ROOT.parents[1] / "work/cluster_disjoint/inputs"
SPLITS = ROOT.parents[1] / "work/cluster_disjoint/run_v2"
NEW_RESULTS = ROOT.parents[1] / "work/cluster_disjoint/notebook_new_run"
if RUN_TRAINING:
    import sys
    previous_argv = sys.argv[:]
    try:
        sys.argv = [str(ROOT / "train.py"), "--inputs", str(INPUTS), "--splits", str(SPLITS), "--out", str(NEW_RESULTS)]
        main()
    finally:
        sys.argv = previous_argv
else:
    print("Training skipped. Original recorded results are in 01_experiment_record.ipynb.")

Training skipped. Original recorded results are in 01_experiment_record.ipynb.


## Audit, clustering, and split implementation
These source displays are for inspection. They do not execute preprocessing or clustering.

In [4]:
from IPython.display import Code, display
for filename in ("prepare.py", "finish.py"):
    print(filename)
    display(Code((ROOT / filename).read_text(), language="python"))

prepare.py


"""Audit source CSVs, cluster full sequences, and freeze component-disjoint splits."""

import argparse, hashlib, json, re, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "drafting_code"))
from snooppi_utils import (
    normalize_split_columns,
    add_pair_ids,
    full_sequence_id,
    file_sha256,
    assert_model_ready,
    preprocessing_report,
)
from snooppi_protocol import mapped_component_frame, mapped_overlap_report

A, B = "partner_A_sequence", "partner_B_sequence"


def dump(obj, path):
    path.write_text(json.dumps(obj, indent=2, default=str) + "\n")


def audit(inputs, out):
    frames = []
    for name in ("train", "validation", "test"):
        p = inputs / f"{name}.csv"
        f = normalize_split_columns(pd.read_csv(p), p)
        f["source_row"] = np.arange(len(f))
        f["original_split"] = name
        frames.append(f)
    f = add_pair_ids(pd.concat(frames, ignore_index=True))
    invalid = pd.Series(False, index=f.index)
    for c in (A, B):
        invalid |= (
            ~f[c]
            .str.upper()
            .str.replace(" ", "", regex=False)
            .str.fullmatch("[ACDEFGHIKLMNPQRSTVWYXBZOUJ]+")
        )
    bad = f[invalid].copy()
    bad["reason"] = "invalid_protein_sequence"
    f = f[~invalid].copy()
    conflict = pd.Series(False, index=f.index)
    for c in ("full_pair_id", "pair_id", "prott5_pair_id"):
        conflict |= f[c].isin(f.groupby(c).label.nunique().loc[lambda x: x > 1].index)
    conflicts = f[conflict].copy()
    conflicts["reason"] = "conflicting_labels_quarantined_without_relabeling"
    f = f[~conflict].copy()
    dup = f[f.full_pair_id.duplicated()].copy()
    dup["reason"] = "consistent_full_pair_duplicate"
    f = f.drop_duplicates("full_pair_id").copy()
    collision = f.pair_id.duplicated(False) | f.prott5_pair_id.duplicated(False)
    coll = f[collision].copy()
    coll["reason"] = "distinct_full_pairs_collide_after_preprocessing"
    f = f[~collision].sort_values("full_pair_id").reset_index(drop=True)
    pd.concat([bad, conflicts, dup, coll]).to_csv(out / "exclusions.csv", index=False)
    f.to_csv(out / "clean_pairs.csv", index=False)
    seqs = {
        full_sequence_id(s): s.upper().replace(" ", "") for c in (A, B) for s in f[c]
    }
    with (out / "proteins.fasta").open("w") as h:
        for k, s in sorted(seqs.items()):
            h.write(f">{k}\n{s}\n")
    record = {
        "source_rows": sum(map(len, frames)),
        "invalid_rows": len(bad),
        "conflict_rows": len(conflicts),
        "duplicate_rows": len(dup),
        "normalized_collision_rows": len(coll),
        "retained_rows": len(f),
        "unique_full_sequences": len(seqs),
        "class_counts": f.label.value_counts().to_dict(),
        "sources": {
            n: file_sha256(inputs / f"{n}.csv") for n in ("train", "validation", "test")
        },
    }
    dump(record, out / "audit.json")
    preprocessing = preprocessing_report({"all": f})
    preprocessing["max_full_sequence_length"] = max(map(len, seqs.values()))
    preprocessing["sequences_over_1024"] = sum(len(s) > 1024 for s in seqs.values())
    dump(preprocessing, out / "preprocessing.json")
    return f, seqs


def assign(f):
    # Fixed deterministic size/class-aware assignment; no model performance is consulted.
    counts = f.groupby("mapped_component").label.agg(["size", "sum"])
    counts["negative"] = counts["size"] - counts["sum"]
    target = (
        np.array([0.8, 0.1, 0.1])[:, None]
        * np.array([len(f), f.label.sum(), (f.label == 0).sum()])[None, :]
    )
    used = np.zeros((3, 3))
    allocation = {}
    for k, r in counts.sort_values(
        ["size", "negative"], ascending=False, kind="stable"
    ).iterrows():
        v = np.array([r["size"], r["sum"], r["negative"]])
        costs = []
        for j in range(3):
            trial = used.copy()
            trial[j] += v
            costs.append(

finish.py


"""Finish a completed clustering; use deterministic edge exclusions if components fail."""

import argparse, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from prepare import assign, dump, A, B
from snooppi_utils import file_sha256, full_sequence_id, assert_model_ready
from snooppi_protocol import UnionFind, mapped_component_frame, mapped_overlap_report


def edge_partition(f, m):
    # Merge clusters sharing an actual model input, but do not merge interaction edges.
    uf = UnionFind()
    lookup = m.set_index("sequence_sha256").cluster_id.to_dict()
    for row in f.itertuples():
        for side in ("A", "B"):
            group = (
                "cluster",
                lookup[full_sequence_id(getattr(row, "partner_" + side + "_sequence"))],
            )
            uf.union(group, ("esm", getattr(row, "protein_" + side + "_id")))
            uf.union(group, ("t5", getattr(row, "prott5_protein_" + side + "_id")))
    roots = sorted({str(uf.find(("cluster", v))) for v in lookup.values()})
    index = {v: i for i, v in enumerate(roots)}
    ids = {s: index[str(uf.find(("cluster", v)))] for s, v in lookup.items()}
    a = f[A].map(full_sequence_id).map(ids).to_numpy()
    b = f[B].map(full_sequence_id).map(ids).to_numpy()
    y = f.label.to_numpy()
    rng = np.random.default_rng(44)
    probs = np.sqrt([0.8, 0.1, 0.1])
    probs /= probs.sum()
    best = None
    for trial in range(256):
        groups = rng.choice(3, size=len(roots), p=probs)
        keep = groups[a] == groups[b]
        spl = groups[a]
        counts = np.array(
            [
                [(keep & (spl == j) & (y == label)).sum() for label in (0, 1)]
                for j in range(3)
            ]
        )
        if counts.min() < 50:
            continue
        fractions = counts.sum(axis=1) / counts.sum()
        if np.max(abs(fractions - [0.8, 0.1, 0.1])) > 0.05:
            continue
        target = np.array([0.8, 0.1, 0.1])[:, None] * counts.sum(axis=0)
        score = np.sum(((counts - target) / np.maximum(target, 1)) ** 2) + 0.1 * (
            1 - keep.mean()
        )
        if best is None or score < best[0]:
            best = (score, trial, keep, spl, groups)
    if best is None:
        raise ValueError(
            "No feasible cluster partition in 256 prespecified assignments"
        )
    _, trial, keep, spl, groups = best
    parts = {
        n: f[keep & (spl == j)].copy()
        for j, n in enumerate(("train", "validation", "test"))
    }
    excluded = f[~keep].copy()
    excluded["reason"] = "partners_assigned_to_different_partitions"
    assignment = pd.DataFrame(
        {
            "sequence_sha256": list(ids),
            "partition": [
                ("train", "validation", "test")[groups[v]] for v in ids.values()
            ],
        }
    )
    return parts, excluded, assignment, trial


def input_isolation(parts):
    from snooppi_utils import sequence_id, prott5_normalize_sequence

    rows = []
    for rule, fn in [
        ("full", full_sequence_id),
        ("esm", sequence_id),
        ("prott5", prott5_normalize_sequence),
    ]:
        keys = {n: {fn(s) for c in (A, B) for s in f[c]} for n, f in parts.items()}
        for i, left in enumerate(keys):
            for right in list(keys)[i + 1 :]:
                rows.append(
                    {
                        "rule": rule,
                        "left": left,
                        "right": right,
                        "overlap": len(keys[left] & keys[right]),
                    }
                )
    result = pd.DataFrame(rows)
    if result.overlap.any():
        raise ValueError("Cross-partition full or model-input protein identity")
    return result


def finish(out):
    if (out / "manifest.json").exists():
        raise FileExistsError("Frozen manifest already exists")
    f = pd.read_csv(out / "clean_pairs.csv")
    m = pd.read_csv(out / "mmseqs2_clusters.csv")
    expected = {full_sequence_id(s) for c